# 🤖 RAG-Based Question Answering with LlamaIndex and Qdrant

## Retrieval-Augmented Generation (RAG) with Vector Search and Reranking

### Project Overview

This project demonstrates an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline for answering questions from a custom knowledge base.

Instead of relying only on the language model's internal knowledge, the system first retrieves relevant information from a document collection and then uses that retrieved context to generate a grounded answer.

The project uses **LlamaIndex** for building the RAG pipeline, **Qdrant** as the vector database, and a **reranking step** to improve the relevance of retrieved information.

### Problem Statement

Traditional language models may provide incorrect or unsupported answers when they do not have access to the required domain-specific information.

A RAG system addresses this problem by:

1. Loading and processing documents
2. Splitting documents into searchable text chunks
3. Creating vector embeddings
4. Storing embeddings in a vector database
5. Retrieving relevant document chunks for a user query
6. Reranking the retrieved results
7. Generating an answer using the most relevant context

### Project Objectives

- Build a complete RAG pipeline
- Understand document ingestion and chunking
- Generate and store vector embeddings
- Perform semantic similarity search
- Use Qdrant for vector storage and retrieval
- Apply reranking to improve retrieved context
- Generate grounded responses using retrieved information
- Evaluate retrieval and answer quality
- Understand practical applications of RAG systems

### Technology Stack

- Python
- LlamaIndex
- Qdrant
- Sentence Transformers
- Vector Embeddings
- Retrieval-Augmented Generation (RAG)
- Reranking
- Large Language Models (LLMs)

### RAG Workflow

**Documents → Chunking → Embeddings → Qdrant Vector Store → Retrieval → Reranking → Relevant Context → LLM → Final Answer**

### Important Note

This project focuses on demonstrating the practical architecture and workflow of a RAG system. The final results and observations are based on the actual experiments performed in this notebook.

### Architecture

**Documents → Chunking → Embeddings → Qdrant Vector Store → Retrieval → Reranking → Relevant Context → LLM Response**

### Technologies

- Python
- LlamaIndex
- Qdrant
- Sentence Transformers
- Hugging Face LLM — TinyLlama 1.1B
- Vector Embeddings
- Retrieval-Augmented Generation (RAG)
- Reranking

In [2]:
import sys
from importlib.metadata import version

print("Python version:", sys.version.split()[0])
print("LlamaIndex version:", version("llama-index"))
print("Qdrant Client version:", version("qdrant-client"))
print("Sentence Transformers version:", version("sentence-transformers"))

print("\nEnvironment ready for RAG pipeline.")

Python version: 3.12.14
LlamaIndex version: 0.14.24
Qdrant Client version: 1.19.0
Sentence Transformers version: 6.0.1

Environment ready for RAG pipeline.


In [3]:
from pathlib import Path

knowledge_base = Path("knowledge_base")
knowledge_base.mkdir(exist_ok=True)

print("Knowledge base folder created successfully.")
print("Location:", knowledge_base.resolve())

Knowledge base folder created successfully.
Location: C:\Users\ILYAS_u6ctnbu\Project_11_RAG_LlamaIndex_Qdrant_Reranking\knowledge_base


In [2]:
from pathlib import Path

# Use the exact project location
project_dir = Path.cwd()
knowledge_dir = project_dir / "knowledge_base"
knowledge_dir.mkdir(parents=True, exist_ok=True)

# Employee handbook content
document = """
EMPLOYEE HANDBOOK

Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.

Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.

Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.

Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

Performance Reviews
Employee performance is generally reviewed periodically by the reporting manager.
Performance discussions may cover goals, achievements, development areas, and future objectives.

Code of Conduct
Employees are expected to maintain professional behavior, respect colleagues,
protect confidential company information, and follow applicable company policies.
"""

# Create the file
file_path = knowledge_dir / "employee_handbook.txt"
file_path.write_text(document.strip(), encoding="utf-8")

# Verify
print("Knowledge-base document created successfully.")
print("File exists:", file_path.exists())
print("File location:", file_path.resolve())
print("File size:", file_path.stat().st_size, "bytes")

Knowledge-base document created successfully.
File exists: True
File location: C:\Users\ILYAS_u6ctnbu\Project_11_RAG_LlamaIndex_Qdrant_Reranking\knowledge_base\employee_handbook.txt
File size: 979 bytes


In [3]:
from llama_index.core import SimpleDirectoryReader

# Load documents from the knowledge base
documents = SimpleDirectoryReader(
    input_dir="knowledge_base"
).load_data()

print("Documents loaded successfully.")
print("Number of documents:", len(documents))

if documents:
    print("\nDocument preview:")
    print(documents[0].text[:500])

Documents loaded successfully.
Number of documents: 1

Document preview:
EMPLOYEE HANDBOOK

Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.

Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.

Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.

Employee Benefits
Employees may be eligible for heal


In [4]:
from llama_index.core.node_parser import SentenceSplitter

# Split the loaded document into smaller searchable chunks
splitter = SentenceSplitter(
    chunk_size=200,
    chunk_overlap=40
)

nodes = splitter.get_nodes_from_documents(documents)

print("Document chunking completed successfully.")
print("Number of chunks:", len(nodes))

print("\nFirst chunk preview:")
print(nodes[0].text)

Document chunking completed successfully.
Number of chunks: 1

First chunk preview:
EMPLOYEE HANDBOOK

Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.

Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.

Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.

Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

Performance Reviews
Employee performance is generally reviewed periodically by the reporting manager.
Performance discussions may cover goals, achievements, development areas, and future objectives.

Code of Conduct
Employees are expected to maintain professional behavior, respect colleagues,
protect confidential company informatio

In [1]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Create the embedding model
embedding_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")
print("Model: sentence-transformers/all-MiniLM-L6-v2")

C:\Users\ILYAS_u6ctnbu\anaconda3\envs\data_science\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ILYAS_u6ctnbu\AppData\Local\llama_index\llama_index\Cache\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<0

Embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2


In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Create an in-memory Qdrant vector database
qdrant_client = QdrantClient(":memory:")

collection_name = "employee_handbook"

# Create collection for 384-dimensional MiniLM embeddings
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Qdrant collection created successfully.")
print("Collection:", collection_name)
print("Vector size: 384")
print("Distance metric: COSINE")

Qdrant collection created successfully.
Collection: employee_handbook
Vector size: 384
Distance metric: COSINE


In [4]:
from pathlib import Path
from qdrant_client.models import PointStruct

# Use the already-created knowledge-base document as the single chunk
file_path = Path("knowledge_base/employee_handbook.txt")
chunk_text = file_path.read_text(encoding="utf-8")

# Generate the embedding
embedding = embedding_model.get_text_embedding(chunk_text)

# Create the Qdrant point
point = PointStruct(
    id=0,
    vector=embedding,
    payload={
        "text": chunk_text,
        "source": "employee_handbook.txt"
    }
)

# Store the vector in the existing collection
qdrant_client.upsert(
    collection_name="employee_handbook",
    points=[point]
)

print("Embedding generated and stored successfully.")
print("Number of vectors stored:", 1)
print("Vector dimension:", len(embedding))
print("Qdrant collection:", "employee_handbook")

Embedding generated and stored successfully.
Number of vectors stored: 1
Vector dimension: 384
Qdrant collection: employee_handbook


In [5]:
# Test semantic search with a user query

query = "What is the company's leave policy?"

# Convert the query into an embedding
query_embedding = embedding_model.get_text_embedding(query)

# Search the Qdrant collection
search_results = qdrant_client.query_points(
    collection_name="employee_handbook",
    query=query_embedding,
    limit=1,
    with_payload=True
)

print("Query:", query)
print("\nRetrieved document:")
print(search_results.points[0].payload["text"])
print("\nSimilarity score:", search_results.points[0].score)

Query: What is the company's leave policy?

Retrieved document:
EMPLOYEE HANDBOOK

Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.

Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.

Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.

Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

Performance Reviews
Employee performance is generally reviewed periodically by the reporting manager.
Performance discussions may cover goals, achievements, development areas, and future objectives.

Code of Conduct
Employees are expected to maintain professional behavior, respect colleagues,
protect confidential company information, and follow applic

In [6]:
# Create searchable sections from the existing handbook document

import re

sections = [
    section.strip()
    for section in re.split(r"\n(?=[A-Z][A-Za-z ]+\n)", chunk_text)
    if section.strip()
]

print("Number of searchable sections:", len(sections))

for i, section in enumerate(sections, start=1):
    print(f"\n--- Section {i} ---")
    print(section[:200])

Number of searchable sections: 7

--- Section 1 ---
EMPLOYEE HANDBOOK

--- Section 2 ---
Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.

--- Section 3 ---
Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.

--- Section 4 ---
Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.

--- Section 5 ---
Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

--- Section 6 ---
Performance Reviews
Employee performance is generally reviewed periodically by the reporting manager.
Performance discussions may cover goals, achievements, development areas, and future objectives.

--- Section 7 ---
Code of Conduct
Employees are expected to maintain profes

In [7]:
# Generate embeddings for all searchable sections

section_embeddings = embedding_model.get_text_embedding_batch(sections)

print("Embedding generation completed successfully.")
print("Number of sections:", len(section_embeddings))
print("Embedding dimension:", len(section_embeddings[0]))

Embedding generation completed successfully.
Number of sections: 7
Embedding dimension: 384


In [8]:
from qdrant_client.models import PointStruct

points = []

for i, (section, embedding) in enumerate(zip(sections, section_embeddings)):
    points.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload={
                "text": section,
                "source": "employee_handbook.txt",
                "section_id": i + 1
            }
        )
    )

qdrant_client.upsert(
    collection_name="employee_handbook",
    points=points
)

print("All embeddings stored successfully.")
print("Vectors stored:", len(points))
print("Collection:", "employee_handbook")

All embeddings stored successfully.
Vectors stored: 7
Collection: employee_handbook


In [9]:
query = "What benefits are available to employees?"

query_embedding = embedding_model.get_text_embedding(query)

search_results = qdrant_client.query_points(
    collection_name="employee_handbook",
    query=query_embedding,
    limit=3,
    with_payload=True
)

print("Query:", query)
print("\n===== TOP 3 RETRIEVED RESULTS =====")

for rank, result in enumerate(search_results.points, start=1):
    print(f"\n--- Result {rank} ---")
    print("Similarity score:", round(result.score, 4))
    print("Section:")
    print(result.payload["text"])

Query: What benefits are available to employees?

===== TOP 3 RETRIEVED RESULTS =====

--- Result 1 ---
Similarity score: 0.5901
Section:
Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

--- Result 2 ---
Similarity score: 0.5475
Section:
EMPLOYEE HANDBOOK

--- Result 3 ---
Similarity score: 0.4296
Section:
Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.


In [10]:
from sentence_transformers import CrossEncoder

# Load a lightweight cross-encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Prepare query-document pairs from the retrieved results
retrieved_points = search_results.points

pairs = [
    (query, result.payload["text"])
    for result in retrieved_points
]

# Calculate reranking scores
rerank_scores = reranker.predict(pairs)

# Combine results with reranking scores
reranked_results = sorted(
    zip(retrieved_points, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Reranking completed successfully.")
print("\n===== RERANKED RESULTS =====")

for rank, (result, score) in enumerate(reranked_results, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Reranking score:", round(float(score), 4))
    print("Section:")
    print(result.payload["text"])

C:\Users\ILYAS_u6ctnbu\anaconda3\envs\data_science\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ILYAS_u6ctnbu\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3438.76it/s]


Reranking completed successfully.

===== RERANKED RESULTS =====

--- Rank 1 ---
Reranking score: 8.1154
Section:
Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

--- Rank 2 ---
Reranking score: -6.8264
Section:
EMPLOYEE HANDBOOK

--- Rank 3 ---
Reranking score: -8.0855
Section:
Company Overview
The company is committed to maintaining a professional, safe, and inclusive workplace.


In [11]:
# Select the top reranked results
top_k_results = reranked_results[:2]

# Build context from the retrieved sections
context = "\n\n".join(
    result.payload["text"]
    for result, score in top_k_results
)

print("Retrieved context prepared successfully.")
print("\n===== CONTEXT =====")
print(context)

Retrieved context prepared successfully.

===== CONTEXT =====
Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

EMPLOYEE HANDBOOK


In [18]:
rag_prompt = f"""
Context:
{context}

Question:
{query}

Based only on the context, answer the question in ONE short sentence.
Return ONLY the answer. Do not include the context, instructions, headings, examples, or explanations.

Answer:
"""

print("Final answer prompt prepared successfully.")
print(rag_prompt)

Final answer prompt prepared successfully.

Context:
Employee Benefits
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.

EMPLOYEE HANDBOOK

Question:
What benefits are available to employees?

Based only on the context, answer the question in ONE short sentence.
Return ONLY the answer. Do not include the context, instructions, headings, examples, or explanations.

Answer:



In [14]:
import torch
from llama_index.llms.huggingface import HuggingFaceLLM

llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=100,
    device_map="cpu",
    model_kwargs={
        "torch_dtype": torch.float32
    }
)

print("LLM loaded successfully.")
print("Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0")
print("Device: CPU")

C:\Users\ILYAS_u6ctnbu\anaconda3\envs\data_science\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ILYAS_u6ctnbu\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading we

LLM loaded successfully.
Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Device: CPU


In [19]:
response = llm.complete(rag_prompt)

print("Corrected RAG answer generated successfully.")
print("\n===== FINAL ANSWER =====")
print(response.text)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Corrected RAG answer generated successfully.

===== FINAL ANSWER =====
Employees may be eligible for health insurance, paid leave, and other benefits according to company policy and employment terms.


In [20]:
test_queries = [
    "What is the leave policy?",
    "What are the working hours?",
    "What is the code of conduct?"
]

print("===== RAG PIPELINE TEST =====")

for test_query in test_queries:
    query_embedding = embedding_model.get_text_embedding(test_query)

    results = qdrant_client.query_points(
        collection_name="employee_handbook",
        query=query_embedding,
        limit=3,
        with_payload=True
    )

    pairs = [
        (test_query, result.payload["text"])
        for result in results.points
    ]

    scores = reranker.predict(pairs)

    best_index = scores.argmax()
    best_result = results.points[best_index]

    print("\nQuestion:", test_query)
    print("Retrieved section:", best_result.payload["text"])
    print("Reranking score:", round(float(scores[best_index]), 4))

print("\nMultiple-query RAG testing completed successfully.")

===== RAG PIPELINE TEST =====

Question: What is the leave policy?
Retrieved section: Leave Policy
Employees should request planned leave in advance through the appropriate company process.
Emergency leave should be communicated to the reporting manager as soon as possible.
Reranking score: 5.0473

Question: What are the working hours?
Retrieved section: Working Hours
Employees are expected to follow the working hours and attendance requirements defined by company policy.
Reranking score: 4.6813

Question: What is the code of conduct?
Retrieved section: Code of Conduct
Employees are expected to maintain professional behavior, respect colleagues,
protect confidential company information, and follow applicable company policies.
Reranking score: 6.5402

Multiple-query RAG testing completed successfully.


In [21]:
evaluation_results = [
    {
        "Question": "What is the leave policy?",
        "Expected Section": "Leave Policy",
        "Retrieved Section": "Leave Policy"
    },
    {
        "Question": "What are the working hours?",
        "Expected Section": "Working Hours",
        "Retrieved Section": "Working Hours"
    },
    {
        "Question": "What is the code of conduct?",
        "Expected Section": "Code of Conduct",
        "Retrieved Section": "Code of Conduct"
    }
]

correct = sum(
    result["Expected Section"] == result["Retrieved Section"]
    for result in evaluation_results
)

total = len(evaluation_results)
retrieval_accuracy = correct / total

print("===== RETRIEVAL EVALUATION =====")
print("Test queries:", total)
print("Correct retrievals:", correct)
print("Retrieval accuracy:", f"{retrieval_accuracy:.2%}")
print("\nAll test queries retrieved the expected sections successfully.")

===== RETRIEVAL EVALUATION =====
Test queries: 3
Correct retrievals: 3
Retrieval accuracy: 100.00%

All test queries retrieved the expected sections successfully.
